In [1]:
# 대학원 입학여부 데이터를 가지고
# Logistic Regression을 Tensorflow Keras로 구현해 보아요!

# 1. 필요한 module import
# 기본 모듈
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 데이터 전처리를 위한  module
# 결측치 처리 => pandas dataframe
# 이상치 처리 => Tukey's Fense방식으로 처리 -> numpy가 필요
#                정규분포상의 zscore값을 이용해서 처리 => scipy
from scipy import stats
# 정규화 처리 => 최대최소값을 이용한 정규화 -> Min-Max Normalization
#                라이브러리를 이용해서 쉽게 정규화 처리를 하고 싶어요!
from sklearn.preprocessing import MinMaxScaler

# Model 구현을 위한 Tensorflow 모듈 import
# Model
from tensorflow.keras.models import Sequential
# Layer
from tensorflow.keras.layers import Flatten, Dense
# Optimizer
from tensorflow.keras.optimizers import SGD


In [3]:
# 1. Raw Data Loading
# 데이터가 있어야 모델을 학습시킬 수 있어요!
# 데이터는 어디서 가져오나요?
# 파일(CSV), Database, Open API(JSON), 직접 입력!
# 최대한 많은 데이터를 확보해야 해요!
# 우리는 지금 Logistic Regression을 하고 있어요!
# 우리 데이터의 종속변수는 0 아니면 1로 표현이 될꺼예요!
# 수집한 데이터에 0의 비율이 90%, 1의 비율이 10% => 이런경우는 문제예요!
# 데이터의 불균형 문제가 발생할 수 있는데 이것도 적절히 처리하면서
# 균형적인 데이터를 최대한 많이 확보할 필요가 있어요!
df = pd.read_csv('/content/admission.csv')
# display(df.head())  # 정상적으로 로딩되었는지 확인!
# print(df.shape) # (400, 4)

(400, 4)


In [21]:
# 2. 데이터 전처리
# 확보한 데이터를 적절하게 전처리를 하지 않으면
# 머신러닝 입력(학습의 용도로)으로 사용할 수 없어요!
# 어떻게 전처리를 하느냐에 따라 모델의 성능이 달라져요!
# 2-1. 결측치 처리(Missing value 처리)
#      삭제(Deletion), 수정(Imputation)
# print(df.isnull().sum(axis=0))
# df.info()
# 결측치가 존재하는지 다양한 방법으로 확인!
# 현재는 결측치가 존재하지 않아요!
# (일반적인 경우는 결측치가 다수 존재)
# 2-2. 이상치 처리(outlier 처리)
# 원래 이상치는 이상한 값을 지칭하는 용어예요!
# 독립변수에 존재하는 이상한 값 => 지대점
# 종속변수에 존재하는 이상한 값 => outlier(이상치)
# 종속변수와 독립변수 상관없이 이상한 값은 모두 이상치(outlier)라고
# 부를꺼예요!
# 눈으로 확인해보면 좋아요!
# boxplot을 이용하면 아주 쉽게 눈으로 이상치의 존재여부를
# 확인할 수 있어요!
# plt.boxplot(df['gre'].values)
# plt.show()
# 이상치가 있는지 확인이 되었으니 이제 실제 이상치 처리를 해 보아요!
# 이상치를 제거할건지 아니면 수정할건지 우리가 선택해야 해요!
# 모든 컬럼(독립변수 & 종속변수)에 대해서 이상치를 제거할꺼예요!
# 이상치 판별하기 위해서 zscore를 이용할꺼예요!
# 각 컬럼에 대해 이상치를 제거한 후 DataFrame을 수정.
# gre 컬럼에 대해서 이상치를 판별해서 이상치를 제거하는 코드를 작성
# zscore_threshold = 2.0

# for col in df.columns:
#     outlier = df[col][np.abs(stats.zscore(df[col])) > zscore_threshold]
#     df = df.loc[~df[col].isin(outlier)]

# print(df.shape) # (382, 4)
# fig = plt.figure()
# ax1 = fig.add_subplot(1,4,1)
# ax2 = fig.add_subplot(1,4,2)
# ax3 = fig.add_subplot(1,4,3)
# ax4 = fig.add_subplot(1,4,4)

# ax1.boxplot(df['admit'])
# ax2.boxplot(df['gre'])
# ax3.boxplot(df['gpa'])
# ax4.boxplot(df['rank'])

# plt.show()

# 2-3. 정규화(Normalization)
# MinMax Normalization을 수행할꺼예요!
# 정규화 하기 전에 독립변수와 종속변수를 분리하는게 편해요!
x_data = df.drop('admit', axis=1, inplace=False).values
t_data = df['admit'].values.reshape(-1,1)

# 정규화를 도와주는 scaler객체를 생성해요!
scaler = MinMaxScaler()
scaler.fit(x_data)  # scaler에게 최대값과 최소값을 알려줘요!
x_data_norm = scaler.transform(x_data)

In [26]:
# 이제 모델 구현을 해 보아요!
# Tensorflow Keras로 구현할때는 모델 그림을 그리면 되요!

model = Sequential()

model.add(Flatten(input_shape=(3,)))
model.add(Dense(units=1,
                activation='sigmoid'))

model.compile(optimizer=SGD(learning_rate=1e-2),
              loss='binary_crossentropy')

model.fit(x_data_norm,
          t_data,
          epochs=500,
          verbose=1)

# 출력되는 내용을 보고 2가지 정도만 일단 확인하면되요!
# loss값이 너무 크지 않은지 확인!
# loss의 최소값은 0인데 가능한 0과 가까운 아주 작은 값일 수록 좋아요!
# 그런데 이게 좀 애매해요!! 출력되는 loss값은
# 학습에 사용되는 데이터값에 따라서 들쭉날쭉해요. 한마디로
# 이 값으로 loss값이 좋다 나쁘다를 말하기 힘들어요!
# 이거보다 더 중요한거는..
# Epoch이 증가할때마다 당연히 loss값은 작아져야 정상!

Epoch 1/500


/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6435  
Epoch 2/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6473 
Epoch 3/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6802 
Epoch 4/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6392  
Epoch 5/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6780 
Epoch 6/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6533 
Epoch 7/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6392  
Epoch 8/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6454 
Epoch 9/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.6767 
Epoch 10/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6153 
Epoch 11/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6587 
Epoch 12/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6664  
Epoch 13/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.6530 
Epoch 14/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.6774 
Epoch 15/500
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.667

In [ ]:
# 학습이 잘 되었다고 가정하면
# 우리모델을 평가해야 해요!
# 우리모델이 정말 잘 만들어졌는지 평가기법을 이용해서 확인!

In [27]:
# 모델이 잘 만들어졌으니 예측해야 해요!
# 예측하고자 하는 데이터는
# np.array([[600, 3.8, 1]])
# 당연히 정규화를 진행한 후 데이터를 입력!
result = model.predict(scaler.transform(np.array([[600, 3.8, 1]])))
print(result) # [[0.36217448]]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
[[0.52319163]]


In [28]:
# Admission 예제를 이용해서
# Logistic Regression을 구현하고
# 성능평가까지 진행해 보아요!
%reset

Once deleted, variables cannot be recovered. Proceed (y/[n])? y


In [30]:
# 필요한 module import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 데이터 전처리 관련
from scipy import stats
from sklearn.preprocessing import MinMaxScaler
# 데이터 분할에 관련된 모듈
from sklearn.model_selection import train_test_split

# Model 구현
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.optimizers import SGD, Adam

In [34]:
# 1. Raw Data Loading
admission = pd.read_csv('/content/admission.csv')
# print(admission.shape)  # (400, 4)
# 기존과는 다르게 독립변수를 2개만 쓸꺼예요!(gre, gpa)
df = admission[['gre', 'gpa', 'admit']]
# display(df.head())

In [36]:
# 2. 데이터 전처리
# 2-1. 결측치 처리
#      현재 우리 데이터에는 결측치 없어요!
# 2-2. 이상치 처리
#      수치적인 이상치는 존재해요.(boxplot으로 확인이 가능)
#      하지만 실제 데이터이기 때문에 이상치로 분류하지 않을꺼예요!
# 2-3. 정규화 처리
#      데이터 분리(독립변수와 종속변수)부터 하는게 편해요!
x_data = df[['gre', 'gpa']].values
t_data = df['admit'].values.reshape(-1,1)
scaler = MinMaxScaler()
scaler.fit(x_data)
x_data_norm = scaler.transform(x_data)
# 2-4. 학습데이터와 평가데이터 분리(확인까지 진행!)
(x_data_train_norm, x_data_test_norm, t_data_train, t_data_test ) = \
train_test_split(x_data_norm,
                 t_data,
                 test_size=0.3,
                 stratify=t_data,
                 random_state=2)

In [ ]:
# Model 구현
model = Sequential()

model.add(Flatten(input_shape=(2,)))
model.add(Dense(units=1,
                activation='sigmoid'))
model.compile(optimizer=Adam(learning_rate=1e-2),
              loss='binary_crossentropy',
              metrics=['accuracy'])
# 이번에는 모델 학습을 진행하면서 validation data를 이용해서
# 각 epoch마다 평가를 진행(평가에 대한 metric은 우리가 지정!)

model.fit(x_data_train_norm,
          t_data_train,
          epochs=300,
          verbose=1,
          validation_split=0.2)
# accuracy: 0.7153 - loss: 0.6148 - val_accuracy: 0.5893 - val_loss: 0.7286